# Classifier training
### PS7 Exoplanet Vetting Pipeline

Builds a labelled feature table from public catalogs (TOI dispositions), trains a calibrated LightGBM multiclass classifier, and reports the confusion matrix, per-class precision/recall, and PR-AUC.

**This downloads a few hundred light curves** â€” run it once; the feature table is cached to `data/features/features.parquet`.

In [ ]:
# --- Setup -------------------------------------------------------------------------
# LOCAL: this notebook imports the `exopipeline` package from the project root.
# COLAB: upload the `exopipeline/` folder (or a zip of it) next to this notebook, then
#        run:  !pip install -q "numpy<2" "astropy<7" lightkurve transitleastsquares \
#                              wotan batman-package emcee corner lightgbm \
#                              scikit-learn imbalanced-learn astroquery pyarrow
import sys, os
for p in [".", "..", r"g:\Exoplanet project"]:
    if os.path.isdir(os.path.join(p, "exopipeline")):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from exopipeline import ingest, detrend, search, vetting, classify, labels, config
print("exopipeline ready")


## 1. Pull the TOI catalog + build a balanced dev sample
Maps `tfopwg_disp` â†’ classes (CP/KP/PC=transit, FP=eclipsing_binary, FA=other). Develop on a **few hundred** bright targets, not the whole archive.

In [ ]:
# CLEAN labels: confirmed planets (transit) + TESS EB catalog (eclipsing_binary)
# + TOI false-alarms (other). This is the label fix that lifts accuracy ~+16 pts
# over the noisy FP->eclipsing_binary mapping (see README ablation).
sample = labels.build_clean_sample(per_class=80, tmag_max=12.5)
print("clean dev sample:", sample["label"].value_counts().to_dict())

## 2. Build the feature table (slow â€” downloads + runs the pipeline per target)
Cached to parquet. Set `REBUILD=False` to reuse an existing table.

In [ ]:
REBUILD = True
if REBUILD:
    df = labels.build_feature_table(sample, max_sectors=3)
else:
    df = classify.load_feature_table()
print(df.shape); df["label"].value_counts()

## 3. Train + calibrate, report metrics

In [ ]:
res = classify.train(df, calibrate=True)
print("PR-AUC (macro):", round(res["pr_auc"], 3))
import pandas as pd
rep = pd.DataFrame(res["report"]).T
print(rep[["precision","recall","f1-score"]].round(3))

In [ ]:
import numpy as np
cm = res["confusion_matrix"]; classes = res["classes"]
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j]>cm.max()/2 else "black")
ax.set(xlabel="Predicted", ylabel="True", title="Confusion matrix")
plt.tight_layout(); plt.show()

## 4. Save the model
`classify.predict` will now use it automatically across the pipeline.

In [ ]:
path = classify.save_model(res["model"])
print("saved ->", path)